# 기업 주력상품 HS + 향후 수출 전망 (+ 품목설명)

이 노트북 **하나만 위에서부터 끝까지 실행**하면, 기업별 주력상품 HS 코드에 **품목설명**과 **향후 수출 전망 증가율**이 붙은 표가 나옵니다.
설정할 건 ⑤의 `BASE_DATE_OVERRIDE` 하나뿐(비우면 자동).

In [1]:
import sys
from pathlib import Path
try:
    current_path = Path(__file__).resolve()
except NameError:
    current_path = Path().resolve()
for parent in current_path.parents:
    if (parent / "stock_forecast" / "DATA").is_dir():
        stock_forecast_path = parent / "stock_forecast"; break
else:
    raise ImportError("stock_forecast/DATA 폴더를 찾을 수 없습니다.")
if str(stock_forecast_path) not in sys.path:
    sys.path.insert(0, str(stock_forecast_path))
print("sys.path 등록:", stock_forecast_path)

sys.path 등록: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\stock_forecast


In [2]:
import re
import pandas as pd
from DATA.stock_invest_function import *

db_info = {'user':'stox7412','password':'Apt106503!~','host':get_db_host(),'port':'3307','database':'investar'}

def normalize_hs6(x):
    """HS 코드를 6자리로 통일 (실수형 표기/선행 0 유실 자동 복원)."""
    if pd.isna(x):
        return None
    s = str(x).strip()
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    s = re.sub(r"\D", "", s)
    if not s:
        return None
    if len(s) % 2 == 1:
        s = "0" + s
    if len(s) < 6:
        return None
    return s[:6]

## HS 코드 → 품목 설명 사전 (Claude 지식 기반)
대화에 나온 코드 + 흔한 중소형 수출품목 + 예시(MLCC, 치과 X-ray)를 담았습니다.
**사전에 없는 코드는 HS 류(2자리) 단위로 자동 설명**됩니다. 새 품목은 `HS6_DESC`에 한 줄 추가하면 됩니다.
※ HS는 2017/2022 개정에 따라 일부 세번이 다를 수 있으니 핵심 종목은 한 번 검증 권장.

In [3]:
HS6_DESC = {
 # 반도체·전자부품
 "854232":"메모리 반도체 (DRAM·HBM 등)", "854231":"시스템반도체 - 프로세서·컨트롤러(CPU/MCU)",
 "854233":"시스템반도체 - 증폭기 IC", "852351":"SSD·플래시메모리 등 반도체 저장장치",
 "847330":"컴퓨터(자동자료처리기계) 부품·부속", "852990":"방송·영상·카메라기기 부품(카메라모듈·안테나 등)",
 "848620":"반도체·디스플레이 제조장비", "841410":"진공펌프",
 "853224":"적층세라믹콘덴서 (MLCC)", "853221":"탄탈 콘덴서", "853210":"전력용 콘덴서(역률보상)",
 "854411":"동 권선(에나멜 동선)", "854519":"탄소·흑연 전극", "850511":"영구자석(금속, NdFeB 등)",
 "851762":"통신·네트워크 장비(송수신·중계기)", "741021":"동박(PCB·배터리용, 적층)",
 # 배터리·2차전지
 "850760":"리튬이온 축전지(배터리 셀)", "850790":"축전지(배터리) 부분품",
 "282520":"리튬 산화물·수산화물(양극재 원료)", "284190":"금속산 염(기타)-리튬금속산화물(양극활물질) 등",
 "760711":"알루미늄박(배터리·포장용)",
 # 광·태양광·LED
 "854141":"발광다이오드(LED)", "854142":"태양전지(셀, 미조립)", "854143":"태양광 모듈·패널",
 # 전력기기
 "853590":"고압 개폐·보호기기(>1000V)", "853720":"고압 배전·제어반(>1000V)",
 "853620":"배선용 회로차단기(<=1000V)", "853890":"개폐·배전기기 부품",
 "850440":"정지형 변환기(인버터·충전기·SMPS)", "850434":"대형 변압기(>500kVA, 초고압 변압기)",
 "851150":"차량용 발전기(올터네이터)·점화기기",
 # 방산
 "930591":"군용 무기 부분품·부속품", "852610":"레이더 장치",
 # 조선·운송장비·자동차
 "890120":"탱커(유조선)", "860310":"전기식 철도 동차(자가추진)", "860900":"운송용 컨테이너",
 "870590":"특수목적 자동차(기타)", "870880":"자동차 서스펜션 시스템·부품", "870829":"자동차 차체 부품(기타)",
 "840820":"차량용 디젤엔진", "841229":"유압식 엔진·모터(기타)", "842123":"엔진용 오일·연료 여과기",
 "401110":"승용차용 타이어(신품)",
 # 기계·공구·금형
 "847982":"혼합·반죽 기계(2차전지 슬러리 등)", "847981":"금속 처리용 기계(권선기 등)",
 "847950":"산업용 로봇(기타)", "844399":"인쇄기 부품·부속", "842489":"분사·살포 기계기구(기타)",
 "848071":"사출·압축 금형(플라스틱·고무용)", "820770":"밀링용 교환 절삭공구",
 "820900":"초경 인서트 팁(공구용)", "680421":"합성다이아몬드 연삭·연마석",
 # 석유화학·플라스틱·고무·화학
 "271019":"석유제품(경유·등유·중유 등, 원유 제외)", "290124":"부타디엔·이소프렌",
 "292910":"이소시아네이트(MDI·TDI)", "390330":"ABS 수지", "390320":"SAN 수지",
 "390290":"폴리프로필렌계 중합체(기타)", "390220":"폴리이소부틸렌", "390130":"에틸렌-비닐아세테이트(EVA)",
 "390761":"PET 수지", "390730":"에폭시 수지", "390799":"폴리에스테르(기타,1차형태)",
 "390690":"아크릴 중합체(기타)", "390390":"스티렌계 중합체(기타)", "391110":"석유수지·쿠마론수지",
 "391190":"석유수지 등(기타,1차형태)", "392190":"플라스틱 판·시트·필름(기타)", "392690":"플라스틱 제품(기타)",
 "400251":"합성고무-NBR 라텍스(니트릴 라텍스)", "400219":"스티렌부타디엔고무(SBR,기타)", "400220":"부타디엔고무(BR)",
 "280700":"황산", "250300":"황", "381700":"혼합 알킬벤젠(LAB, 세제원료)", "382600":"바이오디젤",
 "291739":"방향족 다가카르복실산(기타)", "293499":"핵산·헤테로고리화합물(의약 중간체 등)",
 # 화장품·생활화학
 "330499":"기초·색조 화장품(기타)", "330790":"향료·화장용 제품(기타, 탈취제 등)",
 "340130":"피부세정용 계면활성 제품(바디워시·클렌저)",
 # 바이오·의료
 "300214":"면역제품(백신·항체 등, 소매포장)", "300490":"의약품(기타, 소매용)",
 "901890":"의료·외과용 기기(기타)", "902213":"치과용 X선 장치(Dental X-ray)",
 "902214":"의료·수의용 X선 장치", "902212":"컴퓨터단층촬영기(CT)",
 # 철강·비철금속
 "730890":"철강 구조물·부분품", "721420":"철·비합금강 봉", "721499":"철강 봉(기타)", "721633":"H형강",
 "720836":"광폭 열연강판(코일)", "720712":"철강 반제품(빌릿·슬래브)", "730629":"유정용 케이싱·튜빙 강관(기타)",
 "730630":"용접 철강관(원형단면,기타)", "730511":"송유관용 대구경 강관(SAW)",
 "730729":"스테인리스 관이음쇠(기타)", "730793":"철강 맞대기용접 관이음쇠",
 "720270":"페로몰리브덴", "720260":"페로니켈",
 "740311":"전기동-정련구리 캐소드", "740200":"미정련 구리(조동)", "740329":"구리합금 잉곳(기타)",
 "740400":"구리 스크랩", "740819":"정련 구리선(기타)", "740811":"정련 구리선(>6mm)",
 "740911":"정련 구리 판·대(코일)", "740921":"황동 판·대(코일)", "740721":"황동 봉·형재",
 "760200":"알루미늄 스크랩", "760120":"알루미늄 합금(미가공)", "760612":"알루미늄합금 판·대",
 "760429":"알루미늄 형재·봉(기타)", "761699":"알루미늄 제품(기타)", "261310":"몰리브덴광(배소)",
 # 귀금속 (가격효과 주의)
 "710691":"은(미가공)", "710692":"은(반제품)", "710813":"금(반제품)", "711011":"백금(미가공·분말)",
 "711019":"백금(반제품)", "711021":"팔라듐(미가공·반제품)", "711292":"백금 스크랩",
 "711319":"귀금속 장신구(기타)", "284329":"은 화합물",
 # 식품·수산
 "190230":"파스타·라면류(조리 안 된 기타)", "030354":"냉동 고등어", "030389":"냉동 어류(기타)",
 "030487":"냉동 참치 필레",
 # 기타
 "481190":"도포·가공 종이(기타)", "490199":"인쇄 서적(기타)", "610910":"면 티셔츠(편물)",
 "690919":"이화학용 세라믹 제품(기타)", "851690":"전열기기 부품", "852349":"광학기록매체(기타)",
 "320730":"액체 광택제(도자기·유리용)", "901320":"레이저 기기(기타)", "903090":"전기 측정·검사기기 부품",
 "903190":"측정·검사기기 부품(기타)", "903149":"광학식 측정·검사기기(기타)",
}

HS_CHAPTER = {
 "03":"어패류","07":"채소","08":"과일·견과","09":"커피·차·향신료","15":"동식물성 유지",
 "16":"육·어류 조제품","17":"당류","19":"곡물·곡분 조제품(면류 등)","20":"채소·과일 조제품",
 "21":"기타 조제식료품","22":"음료·주류","23":"사료","24":"담배","25":"소금·황·토석","26":"광·슬래그",
 "27":"광물성 연료(석유)","28":"무기화학품","29":"유기화학품","30":"의료용품","31":"비료","32":"염료·도료",
 "33":"화장품·향료","34":"비누·계면활성제","35":"단백질·접착제·효소","38":"각종 화학공업품","39":"플라스틱",
 "40":"고무","44":"목재","47":"펄프","48":"지·판지","49":"인쇄물","52":"면","54":"인조필라멘트",
 "55":"인조스테이플","59":"공업용 직물","60":"편물","61":"의류(편물)","62":"의류(직물)","63":"기타 섬유제품",
 "68":"석·시멘트 제품","69":"도자제품","70":"유리","71":"귀금속·보석","72":"철강","73":"철강 제품",
 "74":"구리","75":"니켈","76":"알루미늄","78":"납","79":"아연","80":"주석","81":"기타 비금속",
 "82":"공구·날붙이","83":"각종 비금속 제품","84":"일반기계","85":"전기·전자기기","86":"철도차량",
 "87":"자동차·부품","88":"항공기","89":"선박","90":"광학·의료·정밀기기","91":"시계","94":"가구·조명",
 "95":"완구·운동용구","96":"잡품",
}

def hs_desc(code6):
    if pd.isna(code6):
        return ""
    if code6 in HS6_DESC:
        return HS6_DESC[code6]
    ch = str(code6)[:2]
    if ch in HS_CHAPTER:
        return f"({ch}류) {HS_CHAPTER[ch]}"
    return ""

print("HS6 설명 사전:", len(HS6_DESC), "건 | 류 폴백:", len(HS_CHAPTER), "개")

# ============================================================
# HS6_DESC 보강 도구 (새 품목 설명을 코드로 안전하게 추가)
# ============================================================
def add_hs_desc(code, desc):
    """HS6_DESC 사전에 새 품목 설명을 추가/갱신.
    - code는 normalize_hs6()로 6자리 정규화 후 키로 사용
    - 이미 있던 설명과 다르면 [갱신], 새로 추가되면 [추가]로 표시
    """
    hs6 = normalize_hs6(code)
    if hs6 is None:
        print(f"[무시] HS코드 정규화 실패: {code}")
        return
    old = HS6_DESC.get(hs6)
    HS6_DESC[hs6] = desc
    if old and old != desc:
        print(f"[갱신] {hs6}: '{old}' -> '{desc}'")
    else:
        print(f"[추가] {hs6}: '{desc}'")

def add_hs_descs_bulk(mapping: dict):
    """{'854233': '시스템반도체 - 증폭기 IC', ...} 형태로 여러 건을 한 번에 추가."""
    for code, desc in mapping.items():
        add_hs_desc(code, desc)

# 사용 예시:
# add_hs_desc('890120', '탱커(유조선) - 설명 보강')
# add_hs_descs_bulk({
#     '740819': '정련 구리선(기타)',
#     '761699': '알루미늄 제품(기타)',
# })


HS6 설명 사전: 139 건 | 류 폴백: 65 개


In [4]:
# ⑤ 유일한 설정 ───────────────────────────────
# None이면 DB의 v2/ensemble 예측 테이블에서 최신 예측 실행일을 자동으로 사용합니다.
# 특정 예측 회차를 고정하고 싶으면 "2026-07-16" 처럼 날짜 문자열을 넣으세요.
BASE_DATE_OVERRIDE = None   # 예: "2026-06-12". None이면 자동
# ──────────────────────────────────────────────


In [5]:
# ============================================================
# 향후 수출 전망 증가율 (HS6별) — v2/ensemble 예측 테이블 기준
# (한국_수출품목_증가율_조회_v2.ipynb 과 동일한 소스 테이블/지표/로직 사용)
#   - 예전 코드: korea_monthly_trade_data_forecast (expDlr_forecast_12m)  -> 구버전
#   - 이번 코드: korea_monthly_trade_forecast_v2   (ensemble_expDlr)      -> 최신
# ============================================================
from sqlalchemy import create_engine, text
from dateutil.relativedelta import relativedelta
import numpy as np
import pymysql

INDICATOR      = "ensemble_expDlr"
FORECAST_TABLE = "korea_monthly_trade_forecast_v2"

def make_engine(db_info: dict):
    url = (
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
        f"@{db_info['host']}:{int(db_info['port'])}/{db_info['database']}?charset=utf8mb4"
    )
    return create_engine(
        url, pool_pre_ping=True, pool_recycle=1800,
        connect_args={"connect_timeout": 10, "read_timeout": 60, "write_timeout": 60},
    )

engine = make_engine(db_info)

def make_pivot_from_trade_forecast_db(engine, forecast_dates, indicator,
                                       table_name=FORECAST_TABLE, aggfunc="last"):
    if not forecast_dates:
        raise ValueError("forecast_dates 리스트가 비어 있습니다.")
    placeholders = ", ".join([f":d{i}" for i in range(len(forecast_dates))])
    params = {f"d{i}": forecast_dates[i] for i in range(len(forecast_dates))}
    params["indicator"] = indicator
    sql = text(f"""
        SELECT date, hs_code, value
        FROM {table_name}
        WHERE forecast_date IN ({placeholders})
          AND indicator = :indicator
    """)
    with engine.connect() as conn:
        df = pd.read_sql(sql, conn, params=params)
    if df.empty:
        raise ValueError("조건에 해당하는 데이터가 없습니다. forecast_dates/indicator를 확인하세요.")
    df["date"] = pd.to_datetime(df["date"])
    df["hs_code"] = df["hs_code"].astype(str)
    pivot_df = (
        df.pivot_table(index="date", columns="hs_code", values="value", aggfunc=aggfunc)
        .sort_index()
    )
    return pivot_df

def build_forecast_pivot(engine, forecast_date, indicator, target_date=None):
    pv = make_pivot_from_trade_forecast_db(engine=engine, forecast_dates=[forecast_date],
                                            indicator=indicator, aggfunc="last")
    if pv.empty:
        return pv
    td = pv.index.max() if target_date is None else pd.to_datetime(target_date)
    if td in pv.index:
        keep = ~pv.loc[td].isna()
        pv_f = pv.loc[:, keep]
        if pv_f.shape[1] == 0:
            pv_f = pv
    else:
        pv_f = pv
    pv_f = pv_f.sort_index()
    pv_f.columns = pv_f.columns.astype(str)
    return pv_f

def get_unique_forecast_dates(db_info, table_name=FORECAST_TABLE):
    conn = pymysql.connect(
        host=db_info["host"], port=int(db_info.get("port", 3306)), user=db_info["user"],
        password=db_info["password"], database=db_info["database"], charset="utf8mb4"
    )
    try:
        sql = f"SELECT DISTINCT forecast_date FROM {table_name} ORDER BY forecast_date"
        df = pd.read_sql(sql, conn)
    finally:
        conn.close()
    return df

def pivot_dataframe(df, index_col='date', column_col='root_hs_code', indicator_col='indicator',
                     value_col='value', indicator_value='expDlr'):
    filtered_df = df[df[indicator_col] == indicator_value].copy()
    return filtered_df.pivot(index=index_col, columns=column_col, values=value_col)

def analyze_export_growth_full(df, base_date, top_n=100000):
    """analyze_export_growth과 동일 로직이되, 병합에 쓸 원본(raw) 숫자 컬럼을 그대로 반환.
    (원본은 화면 출력용으로 문자열 포맷팅을 하지만, 여기서는 merge를 위해 숫자형을 유지)"""
    df = df.copy()
    base_date = pd.to_datetime(base_date)
    past_start   = base_date - relativedelta(months=12) + relativedelta(days=1)
    past_end     = base_date
    future_start = base_date + relativedelta(days=1)
    future_end   = base_date + relativedelta(months=12)

    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    past_mask = (df.index >= past_start) & (df.index <= past_end)
    past_12m_sum = df.loc[past_mask].sum()
    future_mask = (df.index >= future_start) & (df.index <= future_end)
    future_12m_sum = df.loc[future_mask].sum()

    growth_rate = []
    for past_val, future_val in zip(past_12m_sum, future_12m_sum):
        if pd.isna(past_val) or pd.isna(future_val):
            growth_rate.append(np.nan)
        elif past_val > 0:
            growth_rate.append(((future_val - past_val) / past_val) * 100)
        else:
            growth_rate.append(np.nan)

    result_df = pd.DataFrame({
        'hs_code': df.columns,
        'past_12m_sum': past_12m_sum.values,
        'future_12m_sum': future_12m_sum.values,
        'growth_rate': growth_rate,
    }).dropna(subset=['growth_rate'])

    result_df = result_df.sort_values('growth_rate', ascending=False).reset_index(drop=True)
    return result_df.head(top_n)

# ---- 예측일 자동 선택 (DB의 forecast_date 목록 기반) ----
_fd_raw  = get_unique_forecast_dates(db_info).dropna()
_fd_list = sorted(
    pd.to_datetime(_fd_raw["forecast_date"]).dt.strftime("%Y-%m-%d").unique().tolist()
)
if not _fd_list:
    raise ValueError("DB에 forecast_date가 없습니다.")

THIS_MONTH_FD = BASE_DATE_OVERRIDE if BASE_DATE_OVERRIDE else _fd_list[-1]
print("[사용 가능한 예측일 목록]:", _fd_list)
print("이번달 예측일 (THIS_MONTH_FD):", THIS_MONTH_FD)

# ---- 이번달 예측 피벗 (완전한 예측 컬럼만 유지) ----
this_month_df_filtered = build_forecast_pivot(engine, THIS_MONTH_FD, INDICATOR)

# ---- 실적(과거) 데이터 ----
trade_data_df = fetch_table_data(db_info, "korea_monthly_trade_data")
history_df = pivot_dataframe(
    trade_data_df, index_col='date', column_col='root_hs_code',
    indicator_col='indicator', value_col='value', indicator_value='expDlr'
)
history_df.index = pd.to_datetime(history_df.index)
history_df = history_df.sort_index()
history_df.columns = history_df.columns.astype(str)

# ---- 핵심 기준일: 실적 마지막 달(= 과거12M/미래12M 분기점) ----
LAST_ACTUAL_DATE = history_df.index.max()
BASE_DATE = LAST_ACTUAL_DATE
print("전망 기준일 (BASE_DATE):", BASE_DATE.date())

# ---- 실적 + 이번달 예측 연속 시계열 ----
pv_forecast_region = this_month_df_filtered.loc[this_month_df_filtered.index > LAST_ACTUAL_DATE]
trade_total_df = pd.concat([history_df, pv_forecast_region], axis=0, join='inner').sort_index()
print("연속 시계열 trade_total_df shape:", trade_total_df.shape)

# ---- 전체 HS코드 대상 과거12M vs 미래12M 증가율 계산 (top_n을 크게 줘서 전량 확보) ----
result_all = analyze_export_growth_full(trade_total_df, BASE_DATE, top_n=100000)

growth6 = result_all.rename(columns={
    'past_12m_sum': 'past_12m',
    'future_12m_sum': 'future_12m',
    'growth_rate': '수출전망증가율(%)',
}).copy()
growth6['hscode_6d'] = growth6['hs_code'].astype(str).map(normalize_hs6)
growth6 = (
    growth6.dropna(subset=['hscode_6d'])
    .sort_values('수출전망증가율(%)', ascending=False)
    .drop_duplicates('hscode_6d')
)
print("전망 HS6 개수:", len(growth6))


[사용 가능한 예측일 목록]: ['2025-10-31', '2025-11-02', '2025-11-03', '2025-11-11', '2025-11-12', '2025-11-13', '2025-12-09', '2025-12-10', '2025-12-12', '2025-12-13', '2026-01-14', '2026-02-20', '2026-02-21', '2026-03-06', '2026-03-07', '2026-03-15', '2026-03-16', '2026-04-13', '2026-05-12', '2026-06-12', '2026-07-16']
이번달 예측일 (THIS_MONTH_FD): 2026-07-16
✅ 'korea_monthly_trade_data' 테이블에서 1428902건의 데이터를 가져왔습니다.
전망 기준일 (BASE_DATE): 2026-06-30
연속 시계열 trade_total_df shape: (258, 152)
전망 HS6 개수: 152


In [6]:
# ============================================================
# 기업-HS 코드 수동 입력 관리 도구
# ============================================================
# 방법 A) 아래 리스트에 튜플을 직접 추가:  (ticker, 기업명, hs_code)
# 방법 B) add_company_hscode(...) / add_company_hscodes_bulk(...) 함수를 사용
#         -> 정규화 실패/중복 여부를 알려주고, 안전하게 manual_entries에 append 합니다.
manual_entries = [
    # ('A031330', '에스에이엠티', '854232'),
]

def add_company_hscode(ticker, name, hs_code, entries=None):
    """기업-HS 매핑 1건을 manual_entries에 안전하게 추가.
    - hs_code는 normalize_hs6()로 6자리 정규화해서 중복 판정에 사용(원본 표기는 그대로 저장)
    - (ticker, 정규화된 hs_code) 조합이 이미 있으면 건너뜀
    """
    if entries is None:
        entries = manual_entries
    hs6 = normalize_hs6(hs_code)
    if hs6 is None:
        print(f"[무시] HS코드 정규화 실패: ticker={ticker}, name={name}, hs_code={hs_code}")
        return entries
    dup = any(t == ticker and normalize_hs6(h) == hs6 for (t, n, h) in entries)
    if dup:
        print(f"[중복, 건너뜀] {name}({ticker}) - {hs_code}")
        return entries
    entries.append((ticker, name, hs_code))
    print(f"[추가됨] {name}({ticker}) - {hs_code} (정규화: {hs6})")
    return entries

def add_company_hscodes_bulk(rows, entries=None):
    """rows: [{'ticker':..., 'name':..., 'hs_code':...}, ...] 형태로 여러 건을 한 번에 추가."""
    if entries is None:
        entries = manual_entries
    for r in rows:
        ticker = r.get('ticker')
        name = r.get('name') or r.get('Name')
        hs_code = r.get('hs_code') or r.get('hscode') or r.get('hs_code_raw')
        add_company_hscode(ticker, name, hs_code, entries)
    return entries

# 사용 예시:
# add_company_hscode('A031330', '에스에이엠티', '854232')
# add_company_hscodes_bulk([
#     {'ticker': 'A005930', 'name': '삼성전자',   'hs_code': '854232'},
#     {'ticker': 'A000660', 'name': 'SK하이닉스', 'hs_code': '852351'},
# ])

# ------------------------------------------------------------
# DB에서 기업-HS 맵을 가져와 manual_entries와 합치고 6자리로 통일
# (manual_entries에 새로 추가한 뒤 이 셀만 다시 실행하면 comp가 갱신됩니다)
# ------------------------------------------------------------
comp_raw = fetch_table_data(db_info, 'korea_company_hscode_map')

def _pick(cols, cands):
    low = {str(c).lower(): c for c in cols}
    for cand in cands:
        if cand.lower() in low:
            return low[cand.lower()]
    return None

col_hs   = _pick(comp_raw.columns, ['hs_code','hscode','hs','root_hs_code','hs_code_6d'])
col_name = _pick(comp_raw.columns, ['Name','company','company_name','기업명','종목명'])
col_tkr  = _pick(comp_raw.columns, ['ticker','Code','code','종목코드'])
assert col_hs and col_name, f"hs_code/Name 컬럼 감지 실패: {list(comp_raw.columns)}"

def build_comp():
    """comp_raw(DB) + manual_entries를 합쳐 comp 데이터프레임을 (재)생성.
    manual_entries나 comp_raw를 바꾼 뒤 이 함수를 다시 호출하면 됩니다."""
    ren = {col_hs: 'hs_code_raw', col_name: 'Name'}
    if col_tkr:
        ren[col_tkr] = 'ticker'
    _comp = comp_raw[[c for c in [col_tkr, col_name, col_hs] if c]].rename(columns=ren).copy()
    if manual_entries:
        _comp = pd.concat(
            [_comp, pd.DataFrame(manual_entries, columns=['ticker', 'Name', 'hs_code_raw'])],
            ignore_index=True
        )
    _comp['hscode_6d'] = _comp['hs_code_raw'].map(normalize_hs6)
    _comp = _comp.dropna(subset=['hscode_6d']).reset_index(drop=True)
    return _comp

comp = build_comp()
print("기업-HS 행수:", len(comp), "| 기업수:", comp['Name'].nunique())


✅ 'korea_company_hscode_map' 테이블에서 762건의 데이터를 가져왔습니다.
기업-HS 행수: 622 | 기업수: 410


In [7]:
comp

,ticker,Name,hs_code_raw,hscode_6d
0,A093370,후성,854321,854321
1,A036490,SK머티리얼즈,281290,281290
2,A104830,원익머트리얼즈,854239,854239
3,A144960,뉴파워프라즈마,854239,854239
4,A036930,주성엔니지어링,847989,847989
...,...,...,...,...
617,A011784,금호석유,4002590000,400259
618,A011785,금호석유,4002110000,400211
619,A178920,PI첨단소재,3916909000,391690
620,A000070,삼양홀딩스,290723,290723


In [11]:
# 결합: 기업 주력상품 + 품목설명 + 향후 수출 전망
final = comp.merge(growth6[['hscode_6d','past_12m','future_12m','수출전망증가율(%)']], on='hscode_6d', how='left')
final['품목설명'] = final['hscode_6d'].map(hs_desc)

cols = [c for c in ['ticker','Name','hs_code_raw','hscode_6d','품목설명',
                    'past_12m','future_12m','수출전망증가율(%)'] if c in final.columns]
final = final[cols].sort_values('수출전망증가율(%)', ascending=False, na_position='last').reset_index(drop=True)
print(f"매칭 성공 {final['수출전망증가율(%)'].notna().sum()} / 전체 {len(final)}")
display(final.head(100))

매칭 성공 254 / 전체 622


,ticker,Name,hs_code_raw,hscode_6d,품목설명,past_12m,future_12m,수출전망증가율(%)
0,A005930,삼성전자,8523511000,852351,SSD·플래시메모리 등 반도체 저장장치,2.662051e+10,9.114355e+10,242.380959
1,A005930,삼성전자,8542321030,854232,메모리 반도체 (DRAM·HBM 등),1.806005e+11,5.060081e+11,180.180846
2,A131970,테스나,854232,854232,메모리 반도체 (DRAM·HBM 등),1.806005e+11,5.060081e+11,180.180846
3,A000660,SK하이닉스,8542321010,854232,메모리 반도체 (DRAM·HBM 등),1.806005e+11,5.060081e+11,180.180846
4,A000660,SK하이닉스,854232,854232,메모리 반도체 (DRAM·HBM 등),1.806005e+11,5.060081e+11,180.180846
...,...,...,...,...,...,...,...,...
95,A123010,폴라리스웍스,8542311090,854231,시스템반도체 - 프로세서·컨트롤러(CPU/MCU),3.927510e+10,4.256481e+10,8.376080
96,A003160,디아이,854231,854231,시스템반도체 - 프로세서·컨트롤러(CPU/MCU),3.927510e+10,4.256481e+10,8.376080
97,A001820,삼화콘덴서,850440,850440,정지형 변환기(인버터·충전기·SMPS),9.859585e+08,1.066149e+09,8.133234
98,A131390,피앤이솔루션,850440,850440,정지형 변환기(인버터·충전기·SMPS),9.859585e+08,1.066149e+09,8.133234


In [13]:
final.head(150)

,ticker,Name,hs_code_raw,hscode_6d,품목설명,past_12m,future_12m,수출전망증가율(%)
0,A005930,삼성전자,8523511000,852351,SSD·플래시메모리 등 반도체 저장장치,2.662051e+10,9.114355e+10,242.380959
1,A005930,삼성전자,8542321030,854232,메모리 반도체 (DRAM·HBM 등),1.806005e+11,5.060081e+11,180.180846
2,A131970,테스나,854232,854232,메모리 반도체 (DRAM·HBM 등),1.806005e+11,5.060081e+11,180.180846
3,A000660,SK하이닉스,8542321010,854232,메모리 반도체 (DRAM·HBM 등),1.806005e+11,5.060081e+11,180.180846
4,A000660,SK하이닉스,854232,854232,메모리 반도체 (DRAM·HBM 등),1.806005e+11,5.060081e+11,180.180846
...,...,...,...,...,...,...,...,...
145,A234100,세원,870829,870829,자동차 차체 부품(기타),2.786482e+09,2.923166e+09,4.905249
146,A093380,풍강,870829,870829,자동차 차체 부품(기타),2.786482e+09,2.923166e+09,4.905249
147,A004020,현대제철,8708290000,870829,자동차 차체 부품(기타),2.786482e+09,2.923166e+09,4.905249
148,A126600,코프라,870829,870829,자동차 차체 부품(기타),2.786482e+09,2.923166e+09,4.905249


## 품목설명 누락 진단 + 수동 입력 반영 (재실행용)
새 기업/HS코드를 추가했거나 HS6_DESC에 설명을 보강했다면, DB를 다시 조회하지 않고 이 셀만 다시 실행해서 `final`을 갱신할 수 있습니다.


In [9]:
# ============================================================
# ① 품목설명이 '류(2자리)' 수준 폴백으로만 표시된 코드 진단
# ============================================================
_missing = (
    final.loc[~final['hscode_6d'].isin(HS6_DESC.keys()), ['hscode_6d', '품목설명']]
    .drop_duplicates()
    .sort_values('hscode_6d')
)
if len(_missing):
    print(f"[품목설명 미등록 {len(_missing)}건] 아래 코드에 상세 설명을 추가해 보세요 (add_hs_desc 사용):")
    display(_missing)
else:
    print("모든 HS코드에 상세 설명이 등록되어 있습니다.")

# ============================================================
# ② manual_entries / HS6_DESC를 수정한 뒤 DB 재조회 없이 재조립
#    (comp_raw, growth6는 이미 메모리에 있으므로 build_comp()만 다시 부르면 됨)
# ============================================================
def rebuild_final():
    """manual_entries 또는 HS6_DESC를 갱신한 뒤 호출하면 comp/final을 다시 만들어 반환."""
    _comp = build_comp()
    _final = _comp.merge(
        growth6[['hscode_6d', 'past_12m', 'future_12m', '수출전망증가율(%)']],
        on='hscode_6d', how='left'
    )
    _final['품목설명'] = _final['hscode_6d'].map(hs_desc)
    cols = [c for c in ['ticker', 'Name', 'hs_code_raw', 'hscode_6d', '품목설명',
                        'past_12m', 'future_12m', '수출전망증가율(%)'] if c in _final.columns]
    _final = _final[cols].sort_values('수출전망증가율(%)', ascending=False, na_position='last').reset_index(drop=True)
    print(f"재조립 완료: 매칭 성공 {_final['수출전망증가율(%)'].notna().sum()} / 전체 {len(_final)}")
    return _comp, _final

# 사용 예시:
# add_company_hscode('A031330', '에스에이엠티', '854232')
# add_hs_desc('854233', '시스템반도체 - 증폭기 IC(보강)')
# comp, final = rebuild_final()
# display(final.head(50))


[품목설명 미등록 260건] 아래 코드에 상세 설명을 추가해 보세요 (add_hs_desc 사용):


,hscode_6d,품목설명
560,019049,
444,030341,(03류) 어패류
445,030342,(03류) 어패류
446,030343,(03류) 어패류
447,030344,(03류) 어패류
...,...,...
384,940340,(94류) 가구·조명
378,940540,(94류) 가구·조명
494,950300,(95류) 완구·운동용구
410,961610,(96류) 잡품


In [10]:
final[final['Name'] == '금호석유']

,ticker,Name,hs_code_raw,hscode_6d,품목설명,past_12m,future_12m,수출전망증가율(%)
55,A011781,금호석유,4002190000,400219,"스티렌부타디엔고무(SBR,기타)",1.135245e+09,1.297235e+09,14.269212
464,A011780,금호석유,2921519090,292151,(29류) 유기화학품,NaN,NaN,NaN
465,A011780,금호석유,381239,381239,(38류) 각종 화학공업품,NaN,NaN,NaN
466,A011780,금호석유,2930904090,293090,(29류) 유기화학품,NaN,NaN,NaN
467,A011780,금호석유,3812301000,381230,(38류) 각종 화학공업품,NaN,NaN,NaN
614,A011780,금호석유,4002510000,400251,합성고무-NBR 라텍스(니트릴 라텍스),NaN,NaN,NaN
615,A011782,금호석유,4002209000,400220,부타디엔고무(BR),NaN,NaN,NaN
616,A011783,금호석유,4002709000,400270,(40류) 고무,NaN,NaN,NaN
617,A011784,금호석유,4002590000,400259,(40류) 고무,NaN,NaN,NaN
618,A011785,금호석유,4002110000,400211,(40류) 고무,NaN,NaN,NaN


In [ ]:
# Excel 저장
import os
from datetime import date
SAVE_DIR = os.environ.get('ANALYSIS_DIR', os.getcwd())
os.makedirs(SAVE_DIR, exist_ok=True)
out_path = os.path.join(SAVE_DIR, f"기업주력상품_수출전망_{date.today():%Y-%m-%d}.xlsx")
final.to_excel(out_path, index=False)
print("저장 완료:", out_path)